# 11_gold_training_dataset.ipynb — Construcción del dataset Gold de entrenamiento

Este notebook crea el primer dataset **Gold** de DeepWave Canarias para entrenamiento de modelos predictivos.

La capa Silver ya contiene fuentes limpias por separado. Gold integra esas fuentes en una tabla final:

```text
zona_id + timestamp
```

Objetivo principal:

```text
Predecir altura significativa de ola futura y riesgo marítimo futuro.
```

Salida principal:

```text
gold/training_dataset/
```

Targets creados:

```text
target_hs_3h
target_hs_6h
target_hs_12h
target_hs_24h

target_risk_3h
target_risk_6h
target_risk_12h
target_risk_24h
```

Configuración inicial recomendada:

```text
Target principal: SIMAR hs
Periodo: 2015-01-01 a 2025-12-31
Resolución: horaria
Split temporal:
  train = 2015-2022
  val   = 2023
  test  = 2024-2025
```

Fuentes usadas en esta primera versión Gold:

```text
- SIMAR ocean_hourly          → target y features de oleaje
- SIMAR meteo_hourly          → viento
- SIMAR ocean_physics         → corrientes, SST, salinidad
- REDMAR tide_hourly          → nivel del mar agregado por isla
- AEMET daily                 → meteorología diaria agregada por isla
- beach_geography             → features geográficas estáticas
- bathymetry_features         → features batimétricas estáticas
```

Fuentes NO usadas directamente para el primer entrenamiento:

```text
- ERA5_PROXY      → proxy, reservado para experimentos
- forecast_gfs    → operativo/inferencia, no entrenamiento histórico
- REDEXT/REDCOS   → validación externa, no target principal
- Copernicus huge → reservado para versión Gold ampliada
```

## Celda 0 — Montar Google Drive

In [3]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Celda 1 — Instalar librerías necesarias

In [4]:
!pip -q install pandas numpy pyarrow tqdm scikit-learn

## Celda 2 — Imports, rutas y configuración

In [5]:
from pathlib import Path
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import json
import shutil
import gc
from tqdm.auto import tqdm

BASE_DIR = Path("/content/drive/MyDrive/AI Projects/DeepWave Canarias")
SILVER_DIR = BASE_DIR / "silver"
GOLD_DIR = BASE_DIR / "gold"

OUT_DIR = GOLD_DIR / "training_dataset"
QC_DIR = GOLD_DIR / "_quality_reports"
META_DIR = GOLD_DIR / "_metadata"

for d in [OUT_DIR, QC_DIR, META_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Configuración Gold inicial.
START_DATE = pd.Timestamp("2015-01-01 00:00:00", tz="UTC")
END_DATE = pd.Timestamp("2025-12-31 23:00:00", tz="UTC")

TARGET_SOURCE = "SIMAR"
HORIZONS_HOURS = [3, 6, 12, 24]
LAGS_HOURS = [1, 3, 6, 12, 24]

# Umbrales de riesgo marítimo por altura significativa.
RISK_THRESHOLDS = {
    "low_max": 1.0,
    "moderate_max": 2.0,
    "high_max": 3.0,
}

# Control de tamaño para pruebas rápidas.
# Deja None para procesar todas las zonas.
MAX_ZONES_FOR_TEST = None

# Guardado.
PARTITION_COLS = ["split", "year"]

print("SILVER_DIR:", SILVER_DIR)
print("GOLD_DIR:", GOLD_DIR)

if not SILVER_DIR.exists():
    raise FileNotFoundError("No existe silver/. Ejecuta y valida primero la capa Silver.")

SILVER_DIR: /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver
GOLD_DIR: /content/drive/MyDrive/AI Projects/DeepWave Canarias/gold


## Celda 3 — Funciones generales de lectura y guardado

In [6]:
def ensure_utc(series):
    return pd.to_datetime(series, utc=True, errors="coerce")


def normalize_columns(df):
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    return df


def source_dir(base_table_dir, source):
    return Path(base_table_dir) / f"source={source}"


def read_partitioned_source(table_name, source, required=True):
    """
    Lee una fuente Silver particionada por source=...
    No intenta contar filas; carga la fuente completa y filtra luego por periodo.
    Para este Gold inicial solo se leen fuentes manejables: SIMAR, REDMAR, AEMET.
    """
    table_dir = SILVER_DIR / table_name
    src_dir = source_dir(table_dir, source)

    if not src_dir.exists():
        if required:
            raise FileNotFoundError(f"No existe {src_dir}")
        print(f"AVISO: no existe {src_dir}")
        return pd.DataFrame()

    try:
        df = pd.read_parquet(src_dir)
    except Exception:
        dataset = ds.dataset(str(src_dir), format="parquet", partitioning="hive")
        df = dataset.to_table().to_pandas()

    df = normalize_columns(df)

    if "source" not in df.columns:
        df["source"] = source

    return df


def filter_period(df, timestamp_col="timestamp", start=START_DATE, end=END_DATE):
    if df.empty:
        return df

    if timestamp_col not in df.columns:
        raise ValueError(f"Falta columna temporal {timestamp_col}")

    df = df.copy()
    df[timestamp_col] = ensure_utc(df[timestamp_col])

    mask = df[timestamp_col].between(start, end)

    return df.loc[mask].copy()


def safe_numeric(df, columns):
    df = df.copy()

    for c in columns:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    return df


def remove_if_exists(path):
    path = Path(path)
    if path.exists():
        if path.is_dir():
            shutil.rmtree(path)
        else:
            path.unlink()


def write_partitioned_parquet(df, root_path, partition_cols):
    root_path = Path(root_path)
    remove_if_exists(root_path)
    root_path.mkdir(parents=True, exist_ok=True)

    table = pa.Table.from_pandas(df, preserve_index=False)

    pq.write_to_dataset(
        table,
        root_path=str(root_path),
        partition_cols=partition_cols,
        compression="snappy",
    )


def memory_report(name, df):
    if df is None or df.empty:
        print(f"{name}: vacío")
        return

    mb = df.memory_usage(deep=True).sum() / 1024 / 1024
    print(f"{name}: shape={df.shape}, memoria≈{mb:.1f} MB")


def select_existing(df, columns):
    return [c for c in columns if c in df.columns]


def mode_or_first(series):
    s = series.dropna()

    if s.empty:
        return np.nan

    mode = s.mode(dropna=True)

    if len(mode):
        return mode.iloc[0]

    return s.iloc[0]

## Celda 4 — Cargar fuentes estáticas

In [7]:
BEACH_PATH = SILVER_DIR / "beach_geography" / "beach_geography.parquet"
BATHY_PATH = SILVER_DIR / "bathymetry_features" / "bathymetry_features.parquet"

if not BEACH_PATH.exists():
    raise FileNotFoundError(f"No existe {BEACH_PATH}")

if not BATHY_PATH.exists():
    raise FileNotFoundError(f"No existe {BATHY_PATH}")

beach = pd.read_parquet(BEACH_PATH)
bathy = pd.read_parquet(BATHY_PATH)

beach = normalize_columns(beach)
bathy = normalize_columns(bathy)

if MAX_ZONES_FOR_TEST is not None:
    allowed_zones = set(beach["zona_id"].dropna().astype(str).head(MAX_ZONES_FOR_TEST))
    beach = beach[beach["zona_id"].astype(str).isin(allowed_zones)].copy()
    bathy = bathy[bathy["zona_id"].astype(str).isin(allowed_zones)].copy()
else:
    allowed_zones = set(beach["zona_id"].dropna().astype(str))

print("beach:", beach.shape)
print("bathy:", bathy.shape)

display(beach.head())
display(bathy.head())

beach: (561, 17)
bathy: (561, 36)


,zona_id,nombre_zona,isla,municipio,lat,lon,tipo_zona,orientacion_costa,exposicion_norte,exposicion_oeste,exposicion_este,exposicion_swell_nw,exposicion_swell_ne,vulnerabilidad_costera,vulnerabilidad_source,spatial_match_isla,spatial_match_municipio
0,CAN_TF_EL_PUERTITO_0,El Puertito,Tenerife,Güímar,28.2923,-16.3766,playa,NW,1,1,0,1,1,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True
1,CAN_EH_LA_RESTINGA,La Restinga,El Hierro,El Pinar de El Hierro,27.6408,-17.9799,playa,W,0,1,0,1,0,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True
2,CAN_EH_ARENAS_BLANCAS,Arenas Blancas,El Hierro,Frontera,27.7667,-18.1218,playa,W,0,1,0,1,0,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True
3,CAN_EH_EL_VERODAL,El Verodal,El Hierro,Frontera,27.7471,-18.1512,playa,W,0,1,0,1,0,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True
4,CAN_EH_CHARCO_AZUL_0,Charco Azul,El Hierro,Frontera,27.7563,-18.0990,playa,W,0,1,0,1,0,no_disponible,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,True


,zona_id,nombre_zona,isla,municipio,lat,lon,orientacion_costa,offshore_bearing_deg,depth_100m,depth_500m,...,depth_100m_flag,depth_500m_flag,depth_1km_flag,depth_2km_flag,mean_depth_1km_flag,slope_0_500m_flag,slope_500m_2km_flag,distance_to_10m_isobath_flag,distance_to_20m_isobath_flag,bathymetry_roughness_flag
0,CAN_TF_EL_PUERTITO_0,El Puertito,Tenerife,Güímar,28.2923,-16.3766,NW,315,0.0,0.0,...,0,0,0,0,0,0,0,0,0,1
1,CAN_EH_LA_RESTINGA,La Restinga,El Hierro,El Pinar de El Hierro,27.6408,-17.9799,W,270,2.0,2.0,...,0,0,0,0,0,0,0,0,0,0
2,CAN_EH_ARENAS_BLANCAS,Arenas Blancas,El Hierro,Frontera,27.7667,-18.1218,W,270,30.0,34.0,...,0,0,0,0,0,0,0,0,0,0
3,CAN_EH_EL_VERODAL,El Verodal,El Hierro,Frontera,27.7471,-18.1512,W,270,17.0,47.0,...,0,0,0,0,0,0,0,0,0,0
4,CAN_EH_CHARCO_AZUL_0,Charco Azul,El Hierro,Frontera,27.7563,-18.0990,W,270,32.0,32.0,...,0,0,0,0,0,0,0,0,0,0


## Celda 5 — Cargar base principal: SIMAR `ocean_hourly`

In [8]:
ocean = read_partitioned_source("ocean_hourly", "SIMAR", required=True)
ocean = filter_period(ocean, "timestamp")

if MAX_ZONES_FOR_TEST is not None:
    ocean = ocean[ocean["zona_id"].astype(str).isin(allowed_zones)].copy()

required_ocean_cols = ["timestamp", "zona_id", "lat", "lon", "hs", "tp", "wave_direction"]

missing_ocean_cols = [c for c in required_ocean_cols if c not in ocean.columns]

if missing_ocean_cols:
    raise ValueError(f"Faltan columnas necesarias en SIMAR ocean_hourly: {missing_ocean_cols}")

numeric_ocean_cols = [
    "hs", "hmax", "tp", "tm02", "wave_direction",
    "swell_height", "swell_period", "swell_direction",
    "wind_wave_height", "wind_wave_period", "stokes_drift",
]

ocean = safe_numeric(ocean, numeric_ocean_cols)

ocean_feature_cols = [
    "timestamp", "zona_id", "lat", "lon",
    "hs", "hmax", "tp", "tm02", "wave_direction",
    "swell_height", "swell_period", "swell_direction",
    "wind_wave_height", "wind_wave_period", "stokes_drift",
]

ocean_feature_cols = select_existing(ocean, ocean_feature_cols)

base = ocean[ocean_feature_cols].copy()

base = base.rename(
    columns={
        "lat": "simar_ocean_lat",
        "lon": "simar_ocean_lon",
        "hs": "simar_hs",
        "hmax": "simar_hmax",
        "tp": "simar_tp",
        "tm02": "simar_tm02",
        "wave_direction": "simar_wave_direction",
        "swell_height": "simar_swell_height",
        "swell_period": "simar_swell_period",
        "swell_direction": "simar_swell_direction",
        "wind_wave_height": "simar_wind_wave_height",
        "wind_wave_period": "simar_wind_wave_period",
        "stokes_drift": "simar_stokes_drift",
    }
)

base = (
    base
    .sort_values(["zona_id", "timestamp"])
    .drop_duplicates(subset=["zona_id", "timestamp"], keep="first")
    .reset_index(drop=True)
)

memory_report("base ocean SIMAR", base)
display(base.head())

del ocean
gc.collect()

base ocean SIMAR: shape=(983328, 15), memoria≈168.9 MB


,timestamp,zona_id,simar_ocean_lat,simar_ocean_lon,simar_hs,simar_hmax,simar_tp,simar_tm02,simar_wave_direction,simar_swell_height,simar_swell_period,simar_swell_direction,simar_wind_wave_height,simar_wind_wave_period,simar_stokes_drift
0,2025-05-06 00:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.97,NaN,9.10,4.07,2,0.44,10.64,306.0,0.42,NaN,NaN
1,2025-05-06 01:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.94,NaN,10.01,4.19,358,0.47,10.57,306.0,0.36,NaN,NaN
2,2025-05-06 02:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.91,NaN,10.01,4.48,352,0.50,10.35,307.0,0.19,NaN,NaN
3,2025-05-06 03:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.90,NaN,12.11,4.78,349,0.52,10.35,307.0,0.05,NaN,NaN
4,2025-05-06 04:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.90,NaN,12.11,4.94,348,0.54,10.28,307.0,0.00,NaN,NaN


61

## Celda 6 — Unir geografía estática

In [9]:
geo_cols = [
    "zona_id",
    "nombre_zona",
    "isla",
    "municipio",
    "lat",
    "lon",
    "tipo_zona",
    "orientacion_costa",
    "exposicion_norte",
    "exposicion_oeste",
    "exposicion_este",
    "exposicion_swell_nw",
    "exposicion_swell_ne",
    "vulnerabilidad_costera",
]

geo_cols = select_existing(beach, geo_cols)

geo = beach[geo_cols].drop_duplicates(subset=["zona_id"]).copy()
geo = geo.rename(columns={"lat": "zona_lat", "lon": "zona_lon"})

base = base.merge(geo, on="zona_id", how="left")

# Coordenada final de la zona: preferimos dim_zone; fallback a SIMAR.
base["lat"] = base["zona_lat"].fillna(base["simar_ocean_lat"])
base["lon"] = base["zona_lon"].fillna(base["simar_ocean_lon"])

if base["isla"].isna().any():
    print("AVISO: algunas filas no tienen isla tras unir beach_geography.")

memory_report("base + geography", base)
display(base.head())

base + geography: shape=(983328, 30), memoria≈536.8 MB


,timestamp,zona_id,simar_ocean_lat,simar_ocean_lon,simar_hs,simar_hmax,simar_tp,simar_tm02,simar_wave_direction,simar_swell_height,...,tipo_zona,orientacion_costa,exposicion_norte,exposicion_oeste,exposicion_este,exposicion_swell_nw,exposicion_swell_ne,vulnerabilidad_costera,lat,lon
0,2025-05-06 00:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.97,NaN,9.10,4.07,2,0.44,...,playa,W,0,1,0,1,0,no_disponible,27.7837,-17.9045
1,2025-05-06 01:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.94,NaN,10.01,4.19,358,0.47,...,playa,W,0,1,0,1,0,no_disponible,27.7837,-17.9045
2,2025-05-06 02:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.91,NaN,10.01,4.48,352,0.50,...,playa,W,0,1,0,1,0,no_disponible,27.7837,-17.9045
3,2025-05-06 03:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.90,NaN,12.11,4.78,349,0.52,...,playa,W,0,1,0,1,0,no_disponible,27.7837,-17.9045
4,2025-05-06 04:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.90,NaN,12.11,4.94,348,0.54,...,playa,W,0,1,0,1,0,no_disponible,27.7837,-17.9045


## Celda 7 — Cargar y unir SIMAR `meteo_hourly`

In [10]:
meteo = read_partitioned_source("meteo_hourly", "SIMAR", required=True)
meteo = filter_period(meteo, "timestamp")

if MAX_ZONES_FOR_TEST is not None:
    meteo = meteo[meteo["zona_id"].astype(str).isin(allowed_zones)].copy()

meteo_cols = [
    "timestamp",
    "zona_id",
    "wind_speed",
    "wind_direction",
    "wind_gust",
    "temperature_air",
    "pressure",
    "precipitation",
    "humidity",
    "u10",
    "v10",
]

meteo_cols = select_existing(meteo, meteo_cols)

meteo = meteo[meteo_cols].copy()

meteo = safe_numeric(
    meteo,
    [c for c in meteo.columns if c not in ["timestamp", "zona_id"]],
)

meteo = (
    meteo
    .sort_values(["zona_id", "timestamp"])
    .drop_duplicates(subset=["zona_id", "timestamp"], keep="first")
)

meteo = meteo.rename(
    columns={
        "wind_speed": "simar_wind_speed",
        "wind_direction": "simar_wind_direction",
        "wind_gust": "simar_wind_gust",
        "temperature_air": "simar_temperature_air",
        "pressure": "simar_pressure",
        "precipitation": "simar_precipitation",
        "humidity": "simar_humidity",
        "u10": "simar_u10",
        "v10": "simar_v10",
    }
)

base = base.merge(meteo, on=["zona_id", "timestamp"], how="left")

memory_report("base + SIMAR meteo", base)
display(base.head())

del meteo
gc.collect()

base + SIMAR meteo: shape=(983328, 39), memoria≈604.3 MB


,timestamp,zona_id,simar_ocean_lat,simar_ocean_lon,simar_hs,simar_hmax,simar_tp,simar_tm02,simar_wave_direction,simar_swell_height,...,lon,simar_wind_speed,simar_wind_direction,simar_wind_gust,simar_temperature_air,simar_pressure,simar_precipitation,simar_humidity,simar_u10,simar_v10
0,2025-05-06 00:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.97,NaN,9.10,4.07,2,0.44,...,-17.9045,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-05-06 01:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.94,NaN,10.01,4.19,358,0.47,...,-17.9045,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-05-06 02:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.91,NaN,10.01,4.48,352,0.50,...,-17.9045,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-05-06 03:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.90,NaN,12.11,4.78,349,0.52,...,-17.9045,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-05-06 04:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.90,NaN,12.11,4.94,348,0.54,...,-17.9045,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


0

## Celda 8 — Cargar y unir SIMAR `ocean_physics`

In [11]:
physics = read_partitioned_source("ocean_physics", "SIMAR", required=True)
physics = filter_period(physics, "timestamp")

if MAX_ZONES_FOR_TEST is not None:
    physics = physics[physics["zona_id"].astype(str).isin(allowed_zones)].copy()

physics_cols = [
    "timestamp",
    "zona_id",
    "current_u",
    "current_v",
    "current_speed",
    "current_direction",
    "sea_surface_temperature",
    "sea_surface_salinity",
]

physics_cols = select_existing(physics, physics_cols)

physics = physics[physics_cols].copy()

physics = safe_numeric(
    physics,
    [c for c in physics.columns if c not in ["timestamp", "zona_id"]],
)

physics = (
    physics
    .sort_values(["zona_id", "timestamp"])
    .drop_duplicates(subset=["zona_id", "timestamp"], keep="first")
)

physics = physics.rename(
    columns={
        "current_u": "simar_current_u",
        "current_v": "simar_current_v",
        "current_speed": "simar_current_speed",
        "current_direction": "simar_current_direction",
        "sea_surface_temperature": "simar_sst",
        "sea_surface_salinity": "simar_salinity",
    }
)

base = base.merge(physics, on=["zona_id", "timestamp"], how="left")

memory_report("base + SIMAR physics", base)
display(base.head())

del physics
gc.collect()

base + SIMAR physics: shape=(983328, 45), memoria≈649.3 MB


,timestamp,zona_id,simar_ocean_lat,simar_ocean_lon,simar_hs,simar_hmax,simar_tp,simar_tm02,simar_wave_direction,simar_swell_height,...,simar_precipitation,simar_humidity,simar_u10,simar_v10,simar_current_u,simar_current_v,simar_current_speed,simar_current_direction,simar_sst,simar_salinity
0,2025-05-06 00:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.97,NaN,9.10,4.07,2,0.44,...,NaN,NaN,NaN,NaN,0.157849,0.266802,0.310,30.61,20.469,36.977
1,2025-05-06 01:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.94,NaN,10.01,4.19,358,0.47,...,NaN,NaN,NaN,NaN,0.130704,0.238538,0.272,28.72,20.476,36.977
2,2025-05-06 02:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.91,NaN,10.01,4.48,352,0.50,...,NaN,NaN,NaN,NaN,0.097896,0.227860,0.248,23.25,20.484,36.977
3,2025-05-06 03:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.90,NaN,12.11,4.78,349,0.52,...,NaN,NaN,NaN,NaN,0.064995,0.229993,0.239,15.78,20.497,36.977
4,2025-05-06 04:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.90,NaN,12.11,4.94,348,0.54,...,NaN,NaN,NaN,NaN,0.035895,0.239323,0.242,8.53,20.509,36.977


0

## Celda 9 — Cargar REDMAR y agregar mareas por isla/hora

In [12]:
tide = read_partitioned_source("tide_hourly", "REDMAR", required=True)
tide = filter_period(tide, "timestamp")

tide_cols = [
    "timestamp",
    "station_id",
    "zona_id",
    "isla",
    "sea_level",
    "astronomical_tide",
    "meteorological_residual",
    "tide_phase",
    "hours_to_high_tide",
    "hours_to_low_tide",
    "daily_tidal_range",
]

tide_cols = select_existing(tide, tide_cols)

tide = tide[tide_cols].copy()

for c in [
    "sea_level", "astronomical_tide", "meteorological_residual",
    "hours_to_high_tide", "hours_to_low_tide", "daily_tidal_range",
]:
    if c in tide.columns:
        tide[c] = pd.to_numeric(tide[c], errors="coerce")

# Si no hay isla en REDMAR, se puede usar zona_id para unir, pero aquí intentamos por isla.
if "isla" not in tide.columns or tide["isla"].isna().all():
    print("AVISO: tide no tiene isla útil. Se intentará unión por zona_id.")
    tide_agg = (
        tide
        .groupby(["timestamp", "zona_id"], as_index=False)
        .agg(
            redmar_sea_level=("sea_level", "mean"),
            redmar_astronomical_tide=("astronomical_tide", "mean") if "astronomical_tide" in tide.columns else ("sea_level", "mean"),
            redmar_meteorological_residual=("meteorological_residual", "mean") if "meteorological_residual" in tide.columns else ("sea_level", "mean"),
            redmar_daily_tidal_range=("daily_tidal_range", "mean") if "daily_tidal_range" in tide.columns else ("sea_level", "mean"),
            redmar_tide_phase=("tide_phase", mode_or_first) if "tide_phase" in tide.columns else ("station_id", mode_or_first),
        )
    )

    base = base.merge(tide_agg, on=["zona_id", "timestamp"], how="left")

else:
    tide_agg = (
        tide
        .groupby(["timestamp", "isla"], as_index=False)
        .agg(
            redmar_sea_level=("sea_level", "mean"),
            redmar_astronomical_tide=("astronomical_tide", "mean") if "astronomical_tide" in tide.columns else ("sea_level", "mean"),
            redmar_meteorological_residual=("meteorological_residual", "mean") if "meteorological_residual" in tide.columns else ("sea_level", "mean"),
            redmar_daily_tidal_range=("daily_tidal_range", "mean") if "daily_tidal_range" in tide.columns else ("sea_level", "mean"),
            redmar_hours_to_high_tide=("hours_to_high_tide", "mean") if "hours_to_high_tide" in tide.columns else ("sea_level", "mean"),
            redmar_hours_to_low_tide=("hours_to_low_tide", "mean") if "hours_to_low_tide" in tide.columns else ("sea_level", "mean"),
            redmar_tide_phase=("tide_phase", mode_or_first) if "tide_phase" in tide.columns else ("station_id", mode_or_first),
        )
    )

    base = base.merge(tide_agg, on=["isla", "timestamp"], how="left")

memory_report("base + REDMAR tide", base)
display(base.head())

del tide, tide_agg
gc.collect()

/tmp/ipykernel_9931/4095955891.py:49: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["timestamp", "isla"], as_index=False)


base + REDMAR tide: shape=(983328, 52), memoria≈739.8 MB


,timestamp,zona_id,simar_ocean_lat,simar_ocean_lon,simar_hs,simar_hmax,simar_tp,simar_tm02,simar_wave_direction,simar_swell_height,...,simar_current_direction,simar_sst,simar_salinity,redmar_sea_level,redmar_astronomical_tide,redmar_meteorological_residual,redmar_daily_tidal_range,redmar_hours_to_high_tide,redmar_hours_to_low_tide,redmar_tide_phase
0,2025-05-06 00:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.97,NaN,9.10,4.07,2,0.44,...,30.61,20.469,36.977,1.466,1.539,-0.074,0.932,9.0,3.0,falling
1,2025-05-06 01:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.94,NaN,10.01,4.19,358,0.47,...,28.72,20.476,36.977,1.269,1.338,-0.069,0.932,8.0,2.0,falling
2,2025-05-06 02:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.91,NaN,10.01,4.48,352,0.50,...,23.25,20.484,36.977,1.131,1.194,-0.064,0.932,7.0,1.0,falling
3,2025-05-06 03:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.90,NaN,12.11,4.78,349,0.52,...,15.78,20.497,36.977,1.075,1.138,-0.063,0.932,6.0,0.0,falling
4,2025-05-06 04:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.90,NaN,12.11,4.94,348,0.54,...,8.53,20.509,36.977,1.114,1.181,-0.067,0.932,5.0,11.0,rising


0

## Celda 10 — Cargar AEMET daily y unir por isla + fecha

In [14]:
aemet = read_partitioned_source("meteo_hourly", "AEMET", required=False)

if not aemet.empty:
    aemet = filter_period(aemet, "timestamp")

    # Asegurar fecha diaria.
    if "date" not in aemet.columns:
        aemet["date"] = ensure_utc(aemet["timestamp"]).dt.date
    else:
        aemet["date"] = pd.to_datetime(aemet["date"], errors="coerce").dt.date

    # Evitar bug de pandas con particiones/categorías.
    if "isla" not in aemet.columns:
        print("AVISO: AEMET no tiene columna isla. Se continuará sin unir AEMET.")
        aemet = pd.DataFrame()
    else:
        aemet["isla"] = (
            aemet["isla"]
            .astype("string")
            .str.strip()
        )

        base["isla"] = (
            base["isla"]
            .astype("string")
            .str.strip()
        )

        # Columnas AEMET disponibles.
        aemet_feature_map = {
            "temperature_air": "aemet_temperature_air",
            "temperature_min": "aemet_temperature_min",
            "temperature_max": "aemet_temperature_max",
            "precipitation": "aemet_precipitation",
            "wind_speed": "aemet_wind_speed",
            "wind_direction": "aemet_wind_direction",
            "wind_gust": "aemet_wind_gust",
            "pressure": "aemet_pressure",
            "humidity": "aemet_humidity",
        }

        keep_cols = ["date", "isla"] + [
            c for c in aemet_feature_map.keys()
            if c in aemet.columns
        ]

        aemet = aemet[keep_cols].copy()

        # Convertir variables numéricas.
        for c in keep_cols:
            if c not in ["date", "isla"]:
                aemet[c] = pd.to_numeric(aemet[c], errors="coerce")

        # Eliminar fechas/islas nulas antes del groupby.
        aemet = aemet.dropna(subset=["date", "isla"]).copy()

        # Crear agregaciones solo con columnas existentes.
        agg_dict = {
            out_col: (in_col, "mean")
            for in_col, out_col in aemet_feature_map.items()
            if in_col in aemet.columns
        }

        if len(agg_dict) == 0:
            print("AVISO: AEMET no tiene columnas numéricas útiles para unir.")
        else:
            # IMPORTANTE:
            # observed=True + reset_index evita el error de longitud con categorías.
            aemet_agg = (
                aemet
                .groupby(["date", "isla"], observed=True, dropna=False)
                .agg(**agg_dict)
                .reset_index()
            )

            # Crear date en base.
            base["date"] = ensure_utc(base["timestamp"]).dt.date

            # Si reejecutas la celda, evitar duplicar columnas aemet_*.
            existing_aemet_cols = [
                c for c in base.columns
                if c.startswith("aemet_")
            ]

            if existing_aemet_cols:
                base = base.drop(columns=existing_aemet_cols)

            base = base.merge(
                aemet_agg,
                on=["date", "isla"],
                how="left",
            )

            print("AEMET agregado:")
            print(aemet_agg.shape)
            display(aemet_agg.head())

            del aemet_agg

        del aemet
        gc.collect()

else:
    print("AVISO: AEMET no disponible. Se continúa sin features AEMET.")
    base["date"] = ensure_utc(base["timestamp"]).dt.date

memory_report("base + AEMET daily", base)
display(base.head())

AEMET agregado:
(23194, 11)


,date,isla,aemet_temperature_air,aemet_temperature_min,aemet_temperature_max,aemet_precipitation,aemet_wind_speed,aemet_wind_direction,aemet_wind_gust,aemet_pressure,aemet_humidity
0,2015-01-01,El Hierro,20.0,18.9,21.1,0.0,6.9,40.0,14.4,1023.35,54.0
1,2015-01-01,Fuerteventura,17.9,16.8,19.0,0.0,4.7,99.0,9.7,1025.75,45.0
2,2015-01-01,Gran Canaria,18.6,16.9,20.2,0.0,3.9,100.0,10.3,1023.25,46.0
3,2015-01-01,La Gomera,17.2,15.1,19.2,0.0,4.2,100.0,15.0,1001.25,50.0
4,2015-01-01,La Palma,19.5,16.9,22.1,0.0,4.4,99.0,9.7,1022.60,58.0


base + AEMET daily: shape=(983328, 62), memoria≈844.8 MB


,timestamp,zona_id,simar_ocean_lat,simar_ocean_lon,simar_hs,simar_hmax,simar_tp,simar_tm02,simar_wave_direction,simar_swell_height,...,date,aemet_temperature_air,aemet_temperature_min,aemet_temperature_max,aemet_precipitation,aemet_wind_speed,aemet_wind_direction,aemet_wind_gust,aemet_pressure,aemet_humidity
0,2025-05-06 00:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.97,NaN,9.10,4.07,2,0.44,...,2025-05-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-05-06 01:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.94,NaN,10.01,4.19,358,0.47,...,2025-05-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-05-06 02:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.91,NaN,10.01,4.48,352,0.50,...,2025-05-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-05-06 03:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.90,NaN,12.11,4.78,349,0.52,...,2025-05-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-05-06 04:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.90,NaN,12.11,4.94,348,0.54,...,2025-05-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Celda 11 — Unir features batimétricas

In [15]:
bathy_cols = [
    "zona_id",
    "depth_100m",
    "depth_500m",
    "depth_1km",
    "depth_2km",
    "mean_depth_1km",
    "slope_0_500m",
    "slope_500m_2km",
    "distance_to_10m_isobath",
    "distance_to_20m_isobath",
    "bathymetry_roughness",
    "offshore_bearing_deg",
]

bathy_cols = select_existing(bathy, bathy_cols)

bathy_small = bathy[bathy_cols].drop_duplicates(subset=["zona_id"]).copy()

base = base.merge(bathy_small, on="zona_id", how="left")

memory_report("base + bathymetry", base)
display(base.head())

base + bathymetry: shape=(983328, 73), memoria≈927.4 MB


,timestamp,zona_id,simar_ocean_lat,simar_ocean_lon,simar_hs,simar_hmax,simar_tp,simar_tm02,simar_wave_direction,simar_swell_height,...,depth_500m,depth_1km,depth_2km,mean_depth_1km,slope_0_500m,slope_500m_2km,distance_to_10m_isobath,distance_to_20m_isobath,bathymetry_roughness,offshore_bearing_deg
0,2025-05-06 00:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.97,NaN,9.10,4.07,2,0.44,...,0.0,0.0,0.0,42.0,0.0,0.0,5457.619908,7939.652956,NaN,270
1,2025-05-06 01:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.94,NaN,10.01,4.19,358,0.47,...,0.0,0.0,0.0,42.0,0.0,0.0,5457.619908,7939.652956,NaN,270
2,2025-05-06 02:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.91,NaN,10.01,4.48,352,0.50,...,0.0,0.0,0.0,42.0,0.0,0.0,5457.619908,7939.652956,NaN,270
3,2025-05-06 03:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.90,NaN,12.11,4.78,349,0.52,...,0.0,0.0,0.0,42.0,0.0,0.0,5457.619908,7939.652956,NaN,270
4,2025-05-06 04:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.90,NaN,12.11,4.94,348,0.54,...,0.0,0.0,0.0,42.0,0.0,0.0,5457.619908,7939.652956,NaN,270


## Celda 12 — Crear features temporales

In [16]:
base["timestamp"] = ensure_utc(base["timestamp"])

base["year"] = base["timestamp"].dt.year.astype("int16")
base["month"] = base["timestamp"].dt.month.astype("int8")
base["dayofyear"] = base["timestamp"].dt.dayofyear.astype("int16")
base["dayofweek"] = base["timestamp"].dt.dayofweek.astype("int8")
base["hour"] = base["timestamp"].dt.hour.astype("int8")
base["is_weekend"] = base["dayofweek"].isin([5, 6]).astype("int8")

base["hour_sin"] = np.sin(2 * np.pi * base["hour"] / 24)
base["hour_cos"] = np.cos(2 * np.pi * base["hour"] / 24)

base["month_sin"] = np.sin(2 * np.pi * base["month"] / 12)
base["month_cos"] = np.cos(2 * np.pi * base["month"] / 12)

base["dayofyear_sin"] = np.sin(2 * np.pi * base["dayofyear"] / 366)
base["dayofyear_cos"] = np.cos(2 * np.pi * base["dayofyear"] / 366)

display(base[["timestamp", "year", "month", "hour", "hour_sin", "hour_cos", "month_sin", "month_cos"]].head())

,timestamp,year,month,hour,hour_sin,hour_cos,month_sin,month_cos
0,2025-05-06 00:00:00+00:00,2025,5,0,0.000000,1.000000,0.5,-0.866025
1,2025-05-06 01:00:00+00:00,2025,5,1,0.258819,0.965926,0.5,-0.866025
2,2025-05-06 02:00:00+00:00,2025,5,2,0.500000,0.866025,0.5,-0.866025
3,2025-05-06 03:00:00+00:00,2025,5,3,0.707107,0.707107,0.5,-0.866025
4,2025-05-06 04:00:00+00:00,2025,5,4,0.866025,0.500000,0.5,-0.866025


## Celda 13 — Crear lags exactos por `zona_id + timestamp`

In [17]:
def add_exact_lags(df, variables, lags_hours):
    df = df.copy()

    for lag in tqdm(lags_hours, desc="Creando lags"):
        lag_cols = ["zona_id", "timestamp"] + [v for v in variables if v in df.columns]
        lag_df = df[lag_cols].copy()

        # Un valor observado en t0 se convierte en lag para t0+lag.
        lag_df["timestamp"] = lag_df["timestamp"] + pd.to_timedelta(lag, unit="h")

        rename_map = {
            v: f"{v}_lag_{lag}h"
            for v in variables
            if v in lag_df.columns
        }

        lag_df = lag_df.rename(columns=rename_map)

        df = df.merge(lag_df, on=["zona_id", "timestamp"], how="left")

    return df


lag_variables = [
    "simar_hs",
    "simar_tp",
    "simar_wave_direction",
    "simar_wind_speed",
    "simar_wind_direction",
    "redmar_sea_level",
    "simar_current_speed",
]

lag_variables = [v for v in lag_variables if v in base.columns]

base = add_exact_lags(base, lag_variables, LAGS_HOURS)

memory_report("base + lags", base)
display(base.head())

Creando lags:   0%|          | 0/5 [00:00<?, ?it/s]

base + lags: shape=(983328, 120), memoria≈1242.4 MB


,timestamp,zona_id,simar_ocean_lat,simar_ocean_lon,simar_hs,simar_hmax,simar_tp,simar_tm02,simar_wave_direction,simar_swell_height,...,simar_wind_direction_lag_12h,redmar_sea_level_lag_12h,simar_current_speed_lag_12h,simar_hs_lag_24h,simar_tp_lag_24h,simar_wave_direction_lag_24h,simar_wind_speed_lag_24h,simar_wind_direction_lag_24h,redmar_sea_level_lag_24h,simar_current_speed_lag_24h
0,2025-05-06 00:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.97,NaN,9.10,4.07,2,0.44,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-05-06 01:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.94,NaN,10.01,4.19,358,0.47,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-05-06 02:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.91,NaN,10.01,4.48,352,0.50,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-05-06 03:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.90,NaN,12.11,4.78,349,0.52,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-05-06 04:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.90,NaN,12.11,4.94,348,0.54,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Celda 14 — Crear features rolling sin fuga temporal

In [18]:
base = base.sort_values(["zona_id", "timestamp"]).reset_index(drop=True)

def add_group_rolling(df, group_col, value_col, window, stat, output_col):
    if value_col not in df.columns:
        df[output_col] = np.nan
        return df

    if stat == "mean":
        df[output_col] = (
            df.groupby(group_col, sort=False)[value_col]
            .transform(lambda s: s.shift(1).rolling(window=window, min_periods=max(1, window // 2)).mean())
        )
    elif stat == "max":
        df[output_col] = (
            df.groupby(group_col, sort=False)[value_col]
            .transform(lambda s: s.shift(1).rolling(window=window, min_periods=max(1, window // 2)).max())
        )
    elif stat == "std":
        df[output_col] = (
            df.groupby(group_col, sort=False)[value_col]
            .transform(lambda s: s.shift(1).rolling(window=window, min_periods=max(2, window // 2)).std())
        )
    else:
        raise ValueError(f"stat no soportado: {stat}")

    return df


rolling_specs = [
    ("simar_hs", 6, "mean", "simar_hs_roll_mean_6h"),
    ("simar_hs", 12, "max", "simar_hs_roll_max_12h"),
    ("simar_hs", 24, "mean", "simar_hs_roll_mean_24h"),
    ("simar_wind_speed", 6, "mean", "simar_wind_speed_roll_mean_6h"),
    ("simar_wind_speed", 12, "max", "simar_wind_speed_roll_max_12h"),
    ("redmar_sea_level", 12, "std", "redmar_sea_level_roll_std_12h"),
]

for value_col, window, stat, output_col in tqdm(rolling_specs, desc="Rolling features"):
    base = add_group_rolling(base, "zona_id", value_col, window, stat, output_col)

memory_report("base + rolling", base)
display(base.head())

Rolling features:   0%|          | 0/6 [00:00<?, ?it/s]

base + rolling: shape=(983328, 126), memoria≈1287.5 MB


,timestamp,zona_id,simar_ocean_lat,simar_ocean_lon,simar_hs,simar_hmax,simar_tp,simar_tm02,simar_wave_direction,simar_swell_height,...,simar_wind_speed_lag_24h,simar_wind_direction_lag_24h,redmar_sea_level_lag_24h,simar_current_speed_lag_24h,simar_hs_roll_mean_6h,simar_hs_roll_max_12h,simar_hs_roll_mean_24h,simar_wind_speed_roll_mean_6h,simar_wind_speed_roll_max_12h,redmar_sea_level_roll_std_12h
0,2025-05-06 00:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.97,NaN,9.10,4.07,2,0.44,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-05-06 01:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.94,NaN,10.01,4.19,358,0.47,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-05-06 02:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.91,NaN,10.01,4.48,352,0.50,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-05-06 03:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.90,NaN,12.11,4.78,349,0.52,...,NaN,NaN,NaN,NaN,0.94,NaN,NaN,NaN,NaN,NaN
4,2025-05-06 04:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.90,NaN,12.11,4.94,348,0.54,...,NaN,NaN,NaN,NaN,0.93,NaN,NaN,NaN,NaN,NaN


## Celda 15 — Crear targets futuros de altura significativa

In [19]:
def add_future_targets(df, target_col, horizons_hours):
    df = df.copy()

    target_source = df[["zona_id", "timestamp", target_col]].copy()

    for h in tqdm(horizons_hours, desc="Creando targets futuros"):
        future = target_source.copy()

        # El dato observado en t+h se asigna a la fila t.
        future["timestamp"] = future["timestamp"] - pd.to_timedelta(h, unit="h")
        future = future.rename(columns={target_col: f"target_hs_{h}h"})

        df = df.merge(future, on=["zona_id", "timestamp"], how="left")

    return df


base = add_future_targets(base, "simar_hs", HORIZONS_HOURS)

target_cols = [f"target_hs_{h}h" for h in HORIZONS_HOURS]

display(base[["zona_id", "timestamp", "simar_hs"] + target_cols].head(20))

Creando targets futuros:   0%|          | 0/4 [00:00<?, ?it/s]

,zona_id,timestamp,simar_hs,target_hs_3h,target_hs_6h,target_hs_12h,target_hs_24h
0,CAN_EH_PUERTO_DE_LA_ESTACA,2025-05-06 00:00:00+00:00,0.97,0.90,0.89,0.92,0.96
1,CAN_EH_PUERTO_DE_LA_ESTACA,2025-05-06 01:00:00+00:00,0.94,0.90,0.89,0.92,0.95
2,CAN_EH_PUERTO_DE_LA_ESTACA,2025-05-06 02:00:00+00:00,0.91,0.90,0.89,0.91,0.94
3,CAN_EH_PUERTO_DE_LA_ESTACA,2025-05-06 03:00:00+00:00,0.90,0.89,0.90,0.92,0.93
4,CAN_EH_PUERTO_DE_LA_ESTACA,2025-05-06 04:00:00+00:00,0.90,0.89,0.91,0.92,0.92
5,CAN_EH_PUERTO_DE_LA_ESTACA,2025-05-06 05:00:00+00:00,0.90,0.89,0.92,0.94,0.92
6,CAN_EH_PUERTO_DE_LA_ESTACA,2025-05-06 06:00:00+00:00,0.89,0.90,0.92,0.97,0.92
7,CAN_EH_PUERTO_DE_LA_ESTACA,2025-05-06 07:00:00+00:00,0.89,0.91,0.92,0.96,0.92
8,CAN_EH_PUERTO_DE_LA_ESTACA,2025-05-06 08:00:00+00:00,0.89,0.92,0.91,0.96,0.91
9,CAN_EH_PUERTO_DE_LA_ESTACA,2025-05-06 09:00:00+00:00,0.90,0.92,0.92,0.95,0.91


## Celda 16 — Crear niveles de riesgo marítimo

In [20]:
def risk_level_from_hs(hs):
    hs = pd.to_numeric(hs, errors="coerce")

    risk = pd.Series(np.nan, index=hs.index, dtype="float64")

    risk.loc[hs < RISK_THRESHOLDS["low_max"]] = 0
    risk.loc[(hs >= RISK_THRESHOLDS["low_max"]) & (hs < RISK_THRESHOLDS["moderate_max"])] = 1
    risk.loc[(hs >= RISK_THRESHOLDS["moderate_max"]) & (hs < RISK_THRESHOLDS["high_max"])] = 2
    risk.loc[hs >= RISK_THRESHOLDS["high_max"]] = 3

    return risk.astype("Int64")


for h in HORIZONS_HOURS:
    hs_col = f"target_hs_{h}h"
    risk_col = f"target_risk_{h}h"
    base[risk_col] = risk_level_from_hs(base[hs_col])

risk_cols = [f"target_risk_{h}h" for h in HORIZONS_HOURS]

display(base[["zona_id", "timestamp"] + target_cols + risk_cols].head())

,zona_id,timestamp,target_hs_3h,target_hs_6h,target_hs_12h,target_hs_24h,target_risk_3h,target_risk_6h,target_risk_12h,target_risk_24h
0,CAN_EH_PUERTO_DE_LA_ESTACA,2025-05-06 00:00:00+00:00,0.90,0.89,0.92,0.96,0,0,0,0
1,CAN_EH_PUERTO_DE_LA_ESTACA,2025-05-06 01:00:00+00:00,0.90,0.89,0.92,0.95,0,0,0,0
2,CAN_EH_PUERTO_DE_LA_ESTACA,2025-05-06 02:00:00+00:00,0.90,0.89,0.91,0.94,0,0,0,0
3,CAN_EH_PUERTO_DE_LA_ESTACA,2025-05-06 03:00:00+00:00,0.89,0.90,0.92,0.93,0,0,0,0
4,CAN_EH_PUERTO_DE_LA_ESTACA,2025-05-06 04:00:00+00:00,0.89,0.91,0.92,0.92,0,0,0,0


## Celda 17 — Split temporal train/val/test

In [21]:
def assign_split(timestamp):
    year = timestamp.year

    if 2015 <= year <= 2022:
        return "train"

    if year == 2023:
        return "val"

    if 2024 <= year <= 2025:
        return "test"

    return "unused"


base["split"] = base["timestamp"].apply(assign_split)

split_summary = (
    base
    .groupby("split", as_index=False)
    .agg(
        rows=("zona_id", "size"),
        timestamp_min=("timestamp", "min"),
        timestamp_max=("timestamp", "max"),
        zones=("zona_id", "nunique"),
    )
)

display(split_summary)

split_summary.to_csv(QC_DIR / "gold_train_val_test_split.csv", index=False)

if set(base["split"].unique()) - {"train", "val", "test", "unused"}:
    raise ValueError("Split desconocido detectado.")

,split,rows,timestamp_min,timestamp_max,zones
0,test,197256,2024-01-01 00:00:00+00:00,2025-12-31 23:00:00+00:00,14
1,train,698712,2015-01-01 00:00:00+00:00,2022-12-31 23:00:00+00:00,10
2,val,87360,2023-01-01 00:00:00+00:00,2023-12-31 23:00:00+00:00,10


## Celda 18 — Limpieza final y columnas de trazabilidad

In [22]:
base["gold_dataset_version"] = "v1_silver_validated_simar_target"
base["target_source"] = TARGET_SOURCE
base["risk_threshold_low_max"] = RISK_THRESHOLDS["low_max"]
base["risk_threshold_moderate_max"] = RISK_THRESHOLDS["moderate_max"]
base["risk_threshold_high_max"] = RISK_THRESHOLDS["high_max"]

# Variable auxiliar: fila usable si tiene al menos un target.
base["has_any_target"] = base[target_cols].notna().any(axis=1)

# Fila usable para modelos de +3h.
base["is_trainable_target_3h"] = base["target_hs_3h"].notna().astype("int8")

# Orden estable.
base = base.sort_values(["split", "zona_id", "timestamp"]).reset_index(drop=True)

memory_report("Gold final en memoria", base)
display(base.head())

Gold final en memoria: shape=(983328, 142), memoria≈1552.5 MB


,timestamp,zona_id,simar_ocean_lat,simar_ocean_lon,simar_hs,simar_hmax,simar_tp,simar_tm02,simar_wave_direction,simar_swell_height,...,target_risk_12h,target_risk_24h,split,gold_dataset_version,target_source,risk_threshold_low_max,risk_threshold_moderate_max,risk_threshold_high_max,has_any_target,is_trainable_target_3h
0,2025-05-06 00:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.97,NaN,9.10,4.07,2,0.44,...,0,0,test,v1_silver_validated_simar_target,SIMAR,1.0,2.0,3.0,True,1
1,2025-05-06 01:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.94,NaN,10.01,4.19,358,0.47,...,0,0,test,v1_silver_validated_simar_target,SIMAR,1.0,2.0,3.0,True,1
2,2025-05-06 02:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.91,NaN,10.01,4.48,352,0.50,...,0,0,test,v1_silver_validated_simar_target,SIMAR,1.0,2.0,3.0,True,1
3,2025-05-06 03:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.90,NaN,12.11,4.78,349,0.52,...,0,0,test,v1_silver_validated_simar_target,SIMAR,1.0,2.0,3.0,True,1
4,2025-05-06 04:00:00+00:00,CAN_EH_PUERTO_DE_LA_ESTACA,27.75,-17.666667,0.90,NaN,12.11,4.94,348,0.54,...,0,0,test,v1_silver_validated_simar_target,SIMAR,1.0,2.0,3.0,True,1


## Celda 19 — Reportes de calidad Gold

In [23]:
gold_training_summary = pd.DataFrame(
    [
        {
            "dataset": "gold_training_dataset",
            "version": "v1",
            "rows": len(base),
            "columns": base.shape[1],
            "zones": base["zona_id"].nunique(),
            "timestamp_min": base["timestamp"].min(),
            "timestamp_max": base["timestamp"].max(),
            "target_source": TARGET_SOURCE,
            "period_start": START_DATE,
            "period_end": END_DATE,
            "horizons_hours": json.dumps(HORIZONS_HOURS),
            "lags_hours": json.dumps(LAGS_HOURS),
            "train_rows": int((base["split"] == "train").sum()),
            "val_rows": int((base["split"] == "val").sum()),
            "test_rows": int((base["split"] == "test").sum()),
            "has_any_target_pct": float(base["has_any_target"].mean() * 100),
            **{f"{c}_missing_pct": float(base[c].isna().mean() * 100) for c in target_cols},
        }
    ]
)

feature_missing_summary = (
    base.isna()
    .mean()
    .mul(100)
    .reset_index()
    .rename(columns={"index": "column", 0: "missing_pct"})
    .sort_values("missing_pct", ascending=False)
)

target_distribution_rows = []

for h in HORIZONS_HOURS:
    risk_col = f"target_risk_{h}h"
    hs_col = f"target_hs_{h}h"

    dist = (
        base
        .groupby(["split", risk_col], dropna=False)
        .size()
        .reset_index(name="rows")
    )

    dist["horizon_hours"] = h
    dist["risk_column"] = risk_col
    target_distribution_rows.append(dist)

target_distribution = pd.concat(target_distribution_rows, ignore_index=True)

target_stats = (
    base[target_cols]
    .describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95])
    .T
    .reset_index()
    .rename(columns={"index": "target"})
)

display(gold_training_summary)
display(feature_missing_summary.head(40))
display(target_distribution)
display(target_stats)

gold_training_summary.to_csv(QC_DIR / "gold_training_summary.csv", index=False)
feature_missing_summary.to_csv(QC_DIR / "gold_feature_missing_summary.csv", index=False)
target_distribution.to_csv(QC_DIR / "gold_target_distribution.csv", index=False)
target_stats.to_csv(QC_DIR / "gold_target_stats.csv", index=False)

,dataset,version,rows,columns,zones,timestamp_min,timestamp_max,target_source,period_start,period_end,horizons_hours,lags_hours,train_rows,val_rows,test_rows,has_any_target_pct,target_hs_3h_missing_pct,target_hs_6h_missing_pct,target_hs_12h_missing_pct,target_hs_24h_missing_pct
0,gold_training_dataset,v1,983328,142,14,2015-01-01 00:00:00+00:00,2025-12-31 23:00:00+00:00,SIMAR,2015-01-01 00:00:00+00:00,2025-12-31 23:00:00+00:00,"[3, 6, 12, 24]","[1, 3, 6, 12, 24]",698712,87360,197256,99.986881,0.081763,0.163526,0.327053,0.379527


,column,missing_pct
5,simar_hmax,100.000000
13,simar_wind_wave_period,100.000000
36,simar_humidity,100.000000
32,simar_wind_gust,100.000000
35,simar_precipitation,100.000000
14,simar_stokes_drift,100.000000
33,simar_temperature_air,100.000000
34,simar_pressure,100.000000
118,redmar_sea_level_lag_24h,30.860811
111,redmar_sea_level_lag_12h,30.823693


,split,target_risk_3h,rows,horizon_hours,risk_column,target_risk_6h,target_risk_12h,target_risk_24h
0,test,0,26262,3,target_risk_3h,<NA>,<NA>,<NA>
1,test,1,104917,3,target_risk_3h,<NA>,<NA>,<NA>
2,test,2,50730,3,target_risk_3h,<NA>,<NA>,<NA>
3,test,3,15134,3,target_risk_3h,<NA>,<NA>,<NA>
4,test,<NA>,213,3,target_risk_3h,<NA>,<NA>,<NA>
5,train,0,122441,3,target_risk_3h,<NA>,<NA>,<NA>
6,train,1,371292,3,target_risk_3h,<NA>,<NA>,<NA>
7,train,2,164473,3,target_risk_3h,<NA>,<NA>,<NA>
8,train,3,39975,3,target_risk_3h,<NA>,<NA>,<NA>
9,train,<NA>,531,3,target_risk_3h,<NA>,<NA>,<NA>


,target,count,mean,std,min,5%,25%,50%,75%,95%,max
0,target_hs_3h,982524.0,1.702060,0.770980,0.03,0.58,1.18,1.61,2.13,3.1,7.83
1,target_hs_6h,981720.0,1.702022,0.770968,0.03,0.58,1.18,1.61,2.13,3.1,7.83
2,target_hs_12h,980112.0,1.701992,0.771031,0.03,0.58,1.18,1.61,2.13,3.1,7.83
3,target_hs_24h,979596.0,1.702114,0.771229,0.03,0.58,1.18,1.61,2.13,3.1,7.83


## Celda 20 — Validaciones finales antes de guardar

In [24]:
required_final_cols = [
    "zona_id",
    "timestamp",
    "split",
    "year",
    "simar_hs",
    "simar_tp",
    "simar_wave_direction",
    "target_hs_3h",
    "target_risk_3h",
    "lat",
    "lon",
    "isla",
]

missing_final = [c for c in required_final_cols if c not in base.columns]

if missing_final:
    raise ValueError(f"Faltan columnas finales requeridas: {missing_final}")

if base.empty:
    raise ValueError("El dataset Gold está vacío.")

if base["zona_id"].isna().any():
    raise ValueError("Hay zona_id nulos.")

if base["timestamp"].isna().any():
    raise ValueError("Hay timestamp nulos.")

if base["simar_hs"].isna().mean() > 0.25:
    raise ValueError("simar_hs tiene más del 25% de nulos. Revisar fuente target.")

if base["target_hs_3h"].isna().mean() > 0.25:
    raise ValueError("target_hs_3h tiene más del 25% de nulos. Revisar creación de targets.")

if (base["split"] == "train").sum() == 0 or (base["split"] == "val").sum() == 0 or (base["split"] == "test").sum() == 0:
    raise ValueError("Algún split train/val/test está vacío.")

duplicate_keys = base.duplicated(subset=["zona_id", "timestamp"]).sum()

if duplicate_keys > 0:
    raise ValueError(f"Hay duplicados por zona_id+timestamp: {duplicate_keys}")

print("Validaciones finales Gold superadas.")

Validaciones finales Gold superadas.


## Celda 21 — Guardar dataset Gold particionado

In [25]:
# Convertir algunos object problemáticos a string controlado.
for c in ["zona_id", "nombre_zona", "isla", "municipio", "tipo_zona", "orientacion_costa", "split"]:
    if c in base.columns:
        base[c] = base[c].astype("string")

# Asegurar year entero.
base["year"] = pd.to_numeric(base["year"], errors="coerce").astype("int16")

write_partitioned_parquet(base, OUT_DIR, PARTITION_COLS)

print("Guardado Gold training_dataset en:")
print(OUT_DIR)

# Guardar lista de columnas.
columns_metadata = pd.DataFrame(
    {
        "column": base.columns,
        "dtype": [str(base[c].dtype) for c in base.columns],
    }
)

columns_metadata.to_csv(META_DIR / "gold_training_dataset_columns.csv", index=False)

with open(META_DIR / "gold_training_config.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "dataset": "gold_training_dataset",
            "version": "v1",
            "target_source": TARGET_SOURCE,
            "start_date": str(START_DATE),
            "end_date": str(END_DATE),
            "horizons_hours": HORIZONS_HOURS,
            "lags_hours": LAGS_HOURS,
            "risk_thresholds": RISK_THRESHOLDS,
            "partition_cols": PARTITION_COLS,
            "notes": "Primera versión Gold basada en SIMAR como target principal y fuentes Silver validadas.",
        },
        f,
        indent=2,
        ensure_ascii=False,
        default=str,
    )

print("Metadata guardada.")

Guardado Gold training_dataset en:
/content/drive/MyDrive/AI Projects/DeepWave Canarias/gold/training_dataset
Metadata guardada.


## Celda 22 — Comprobación final de lectura

In [26]:
dataset = ds.dataset(str(OUT_DIR), format="parquet", partitioning="hive")
row_count = dataset.count_rows()

sample = dataset.head(10).to_pandas()

print("Filas guardadas:", row_count)
print("Columnas:", len(dataset.schema.names))
display(sample)

if row_count == 0:
    raise ValueError("No se guardó ninguna fila en Gold training_dataset.")

if row_count != len(base):
    print("AVISO: row_count guardado no coincide exactamente con len(base). Revisar particiones.")
else:
    print("Row count guardado coincide con dataset en memoria.")

print("\nReportes Gold:")
for p in sorted(QC_DIR.glob("gold_*.csv")):
    print("-", p)

print("\nMetadata Gold:")
for p in sorted(META_DIR.glob("gold_training*")):
    print("-", p)

print("\n✅ Gold training_dataset v1 generado correctamente.")

Filas guardadas: 983328
Columnas: 142


,timestamp,zona_id,simar_ocean_lat,simar_ocean_lon,simar_hs,simar_hmax,simar_tp,simar_tm02,simar_wave_direction,simar_swell_height,...,target_risk_24h,gold_dataset_version,target_source,risk_threshold_low_max,risk_threshold_moderate_max,risk_threshold_high_max,has_any_target,is_trainable_target_3h,split,year
0,2024-01-01 00:00:00+00:00,CAN_FV_GRAN_TARAJAL,28.166667,-14.0,0.14,NaN,14.66,2.49,129,0.06,...,0,v1_silver_validated_simar_target,SIMAR,1.0,2.0,3.0,True,1,test,2024
1,2024-01-01 01:00:00+00:00,CAN_FV_GRAN_TARAJAL,28.166667,-14.0,0.14,NaN,14.66,2.57,270,0.06,...,0,v1_silver_validated_simar_target,SIMAR,1.0,2.0,3.0,True,1,test,2024
2,2024-01-01 02:00:00+00:00,CAN_FV_GRAN_TARAJAL,28.166667,-14.0,0.14,NaN,14.66,2.63,283,0.07,...,0,v1_silver_validated_simar_target,SIMAR,1.0,2.0,3.0,True,1,test,2024
3,2024-01-01 03:00:00+00:00,CAN_FV_GRAN_TARAJAL,28.166667,-14.0,0.15,NaN,14.66,2.53,303,0.07,...,0,v1_silver_validated_simar_target,SIMAR,1.0,2.0,3.0,True,1,test,2024
4,2024-01-01 04:00:00+00:00,CAN_FV_GRAN_TARAJAL,28.166667,-14.0,0.17,NaN,16.12,2.30,32,0.08,...,0,v1_silver_validated_simar_target,SIMAR,1.0,2.0,3.0,True,1,test,2024
5,2024-01-01 05:00:00+00:00,CAN_FV_GRAN_TARAJAL,28.166667,-14.0,0.19,NaN,16.12,2.34,58,0.09,...,0,v1_silver_validated_simar_target,SIMAR,1.0,2.0,3.0,True,1,test,2024
6,2024-01-01 06:00:00+00:00,CAN_FV_GRAN_TARAJAL,28.166667,-14.0,0.19,NaN,16.12,2.50,62,0.10,...,0,v1_silver_validated_simar_target,SIMAR,1.0,2.0,3.0,True,1,test,2024
7,2024-01-01 07:00:00+00:00,CAN_FV_GRAN_TARAJAL,28.166667,-14.0,0.18,NaN,16.12,2.61,52,0.10,...,0,v1_silver_validated_simar_target,SIMAR,1.0,2.0,3.0,True,1,test,2024
8,2024-01-01 08:00:00+00:00,CAN_FV_GRAN_TARAJAL,28.166667,-14.0,0.18,NaN,16.12,2.75,242,0.11,...,0,v1_silver_validated_simar_target,SIMAR,1.0,2.0,3.0,True,1,test,2024
9,2024-01-01 09:00:00+00:00,CAN_FV_GRAN_TARAJAL,28.166667,-14.0,0.18,NaN,16.12,2.87,240,0.11,...,0,v1_silver_validated_simar_target,SIMAR,1.0,2.0,3.0,True,1,test,2024


Row count guardado coincide con dataset en memoria.

Reportes Gold:
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/gold/_quality_reports/gold_feature_missing_summary.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/gold/_quality_reports/gold_target_distribution.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/gold/_quality_reports/gold_target_stats.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/gold/_quality_reports/gold_train_val_test_split.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/gold/_quality_reports/gold_training_summary.csv

Metadata Gold:
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/gold/_metadata/gold_training_config.json
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/gold/_metadata/gold_training_dataset_columns.csv

✅ Gold training_dataset v1 generado correctamente.


## Resultado esperado

Al final debe aparecer:

```text
Validaciones finales Gold superadas.
✅ Gold training_dataset v1 generado correctamente.
```

Salidas:

```text
gold/training_dataset/split=train/year=YYYY/*.parquet
gold/training_dataset/split=val/year=2023/*.parquet
gold/training_dataset/split=test/year=2024/*.parquet
gold/training_dataset/split=test/year=2025/*.parquet
```

Reportes:

```text
gold/_quality_reports/gold_training_summary.csv
gold/_quality_reports/gold_feature_missing_summary.csv
gold/_quality_reports/gold_target_distribution.csv
gold/_quality_reports/gold_target_stats.csv
gold/_quality_reports/gold_train_val_test_split.csv
```

Metadata:

```text
gold/_metadata/gold_training_config.json
gold/_metadata/gold_training_dataset_columns.csv
```

Este dataset es la primera base para entrenar modelos de:

```text
- regresión: target_hs_3h/6h/12h/24h
- clasificación: target_risk_3h/6h/12h/24h
```